# Summarization Models Evaluation using ROUGE Scores (Original and V2)

This notebook evaluates the quality of TextRank, LexRank, DistilBART, and Pegasus summaries (original and v2 versions) using ROUGE (Recall-Oriented Understudy for Gisting Evaluation) metrics against ground truth summaries.

## Objective:
- Load overall summaries from `OP_SUMMARIES/` folder (TextRank, LexRank, DistilBART, Pegasus - original and v2)
- Load ground truth summaries from `ground_truth/` folder
- Calculate ROUGE-1, ROUGE-2, and ROUGE-L scores for all models (original and v2)
- Compare performance across all summarization models and versions
- Generate comprehensive evaluation report

## ROUGE Metrics:
- **ROUGE-1**: Measures overlap of unigrams (single words)
- **ROUGE-2**: Measures overlap of bigrams (word pairs)
- **ROUGE-L**: Measures longest common subsequence

## Input:
- `OP_SUMMARIES/` folder containing overall summaries:
  - Original: `textrank_overall_summary.txt`, `lexrank_overall_summary.txt`, `distilbart_overall_summary.txt`, `pegasus_overall_summary.txt`
  - V2: `textrank_v2_overall_summary.txt`, `lexrank_v2_overall_summary.txt`, `distilbart_v2_overall_summary.txt`
- `ground_truth/` folder containing reference summaries

## Output:
- ROUGE scores comparison table for all models (original and v2)
- Performance analysis comparing original vs v2 versions
- Recommendations


In [1]:
pip install rouge-score

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import required libraries
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ROUGE evaluation
try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
    print("Using rouge_score library")
except ImportError:
    ROUGE_AVAILABLE = False
    print("rouge_score not available, will use alternative implementation")

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

print("Libraries imported successfully")


Using rouge_score library
Libraries imported successfully


In [3]:
# Configuration - resolve project root (notebook lives in eval/ subfolder)
if (Path.cwd() / "OP_SUMMARIES").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "OP_SUMMARIES").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
EVAL_OUTPUT_DIR = PROJECT_ROOT / "eval"

op_summaries_path = PROJECT_ROOT / "OP_SUMMARIES"
ground_truth_path = PROJECT_ROOT / "ground_truth"

# Initialize ROUGE scorer
if ROUGE_AVAILABLE:
    rouge_scorer_instance = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
else:
    rouge_scorer_instance = None

print("Configuration:")
print(f"OP Summaries path: {op_summaries_path}")
print(f"Ground truth path: {ground_truth_path}")
print(f"ROUGE library available: {ROUGE_AVAILABLE}")

# Check if paths exist
print(f"\nPath verification:")
print(f"OP Summaries exist: {op_summaries_path.exists()}")
print(f"Ground truth exists: {ground_truth_path.exists()}")


Configuration:
OP Summaries path: OP_SUMMARIES
Ground truth path: ground_truth
ROUGE library available: True

Path verification:
OP Summaries exist: True
Ground truth exists: True


In [4]:
# Load summaries and ground truth
def load_summary(file_path):
    """Load summary from file (OP_SUMMARIES files are already cleaned, no headers)"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            return content
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return ""

def load_ground_truth():
    """Load ground truth files"""
    ground_truth_files = {}
    
    # Load GPT-4o-mini ground truth
    gpt_file = ground_truth_path / "gpt-4o-mini.txt"
    if gpt_file.exists():
        ground_truth_files['gpt-4o-mini'] = load_summary(gpt_file)
        print(f"Loaded GPT-4o-mini ground truth")
    
    # Load Groq-LLaMA ground truth
    groq_file = ground_truth_path / "groq-llama.txt"
    if groq_file.exists():
        ground_truth_files['groq-llama'] = load_summary(groq_file)
        print(f"Loaded Groq-LLaMA ground truth")
    
    return ground_truth_files

# Load all summaries from OP_SUMMARIES folder
print("Loading summaries from OP_SUMMARIES folder...")

# Load TextRank overall summary
textrank_file = op_summaries_path / "textrank_overall_summary.txt"
textrank_summary = ""
if textrank_file.exists():
    textrank_summary = load_summary(textrank_file)
    print(f"Loaded TextRank overall summary ({len(textrank_summary)} characters)")
else:
    print(f"Warning: TextRank summary not found")

# Load LexRank overall summary
lexrank_file = op_summaries_path / "lexrank_overall_summary.txt"
lexrank_summary = ""
if lexrank_file.exists():
    lexrank_summary = load_summary(lexrank_file)
    print(f"Loaded LexRank overall summary ({len(lexrank_summary)} characters)")
else:
    print(f"Warning: LexRank summary not found")

# Load DistilBART overall summary
distilbart_file = op_summaries_path / "distilbart_overall_summary.txt"
distilbart_summary = ""
if distilbart_file.exists():
    distilbart_summary = load_summary(distilbart_file)
    print(f"Loaded DistilBART overall summary ({len(distilbart_summary)} characters)")
else:
    print(f"Warning: DistilBART summary not found")

# Load Pegasus overall summary
pegasus_file = op_summaries_path / "pegasus_overall_summary.txt"
pegasus_summary = ""
if pegasus_file.exists():
    pegasus_summary = load_summary(pegasus_file)
    print(f"Loaded Pegasus overall summary ({len(pegasus_summary)} characters)")
else:
    print(f"Warning: Pegasus summary not found")

# Load ground truth
ground_truth_summaries = load_ground_truth()

print(f"\nSummary loading completed:")
print(f"TextRank: {'Loaded' if textrank_summary else 'Not found'}")
print(f"LexRank: {'Loaded' if lexrank_summary else 'Not found'}")
print(f"DistilBART: {'Loaded' if distilbart_summary else 'Not found'}")
print(f"Pegasus: {'Loaded' if pegasus_summary else 'Not found'}")
print(f"Ground truth summaries loaded: {len(ground_truth_summaries)}")


Loading summaries from OP_SUMMARIES folder...
Loaded TextRank overall summary (751 characters)
Loaded LexRank overall summary (485 characters)
Loaded DistilBART overall summary (977 characters)
Loaded Pegasus overall summary (1718 characters)
Loaded GPT-4o-mini ground truth
Loaded Groq-LLaMA ground truth

Summary loading completed:
TextRank: Loaded
LexRank: Loaded
DistilBART: Loaded
Pegasus: Loaded
Ground truth summaries loaded: 2


In [5]:
# Calculate ROUGE scores
def calculate_rouge_scores_simple(predicted, reference):
    """Simple ROUGE implementation using basic text overlap"""
    if not predicted or not reference:
        return {
            'rouge1': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
            'rouge2': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
            'rougeL': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0}
        }
    
    # Tokenize texts
    pred_tokens = word_tokenize(predicted.lower())
    ref_tokens = word_tokenize(reference.lower())
    
    # ROUGE-1 (unigrams)
    pred_unigrams = set(pred_tokens)
    ref_unigrams = set(ref_tokens)
    
    overlap_1 = len(pred_unigrams.intersection(ref_unigrams))
    precision_1 = overlap_1 / len(pred_unigrams) if len(pred_unigrams) > 0 else 0
    recall_1 = overlap_1 / len(ref_unigrams) if len(ref_unigrams) > 0 else 0
    f1_1 = 2 * precision_1 * recall_1 / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0
    
    # ROUGE-2 (bigrams)
    pred_bigrams = set(zip(pred_tokens, pred_tokens[1:]))
    ref_bigrams = set(zip(ref_tokens, ref_tokens[1:]))
    
    overlap_2 = len(pred_bigrams.intersection(ref_bigrams))
    precision_2 = overlap_2 / len(pred_bigrams) if len(pred_bigrams) > 0 else 0
    recall_2 = overlap_2 / len(ref_bigrams) if len(ref_bigrams) > 0 else 0
    f1_2 = 2 * precision_2 * recall_2 / (precision_2 + recall_2) if (precision_2 + recall_2) > 0 else 0
    
    # ROUGE-L (simplified - using word overlap ratio)
    f1_L = f1_1  # Simplified version
    
    return {
        'rouge1': {'precision': precision_1, 'recall': recall_1, 'fmeasure': f1_1},
        'rouge2': {'precision': precision_2, 'recall': recall_2, 'fmeasure': f1_2},
        'rougeL': {'precision': precision_1, 'recall': recall_1, 'fmeasure': f1_L}
    }

def calculate_rouge_scores(predicted, reference):
    """Calculate ROUGE scores for predicted vs reference text"""
    if not predicted or not reference:
        return {
            'rouge1': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
            'rouge2': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
            'rougeL': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0}
        }
    
    if ROUGE_AVAILABLE and rouge_scorer_instance:
        scores = rouge_scorer_instance.score(reference, predicted)
        return scores
    else:
        return calculate_rouge_scores_simple(predicted, reference)

# Calculate ROUGE scores for overall summaries against ground truth
rouge_results = []

print("Calculating ROUGE scores for overall summaries...")

# Evaluate all summaries against both ground truth files
for gt_name, ground_truth in ground_truth_summaries.items():
    print(f"\nEvaluating against {gt_name} ground truth:")
    
    # Calculate TextRank ROUGE scores
    textrank_scores = None
    if textrank_summary:
        textrank_scores = calculate_rouge_scores(textrank_summary, ground_truth)
        print(f"  TextRank ROUGE-1 F1: {textrank_scores['rouge1'].fmeasure:.3f}")
        print(f"  TextRank ROUGE-2 F1: {textrank_scores['rouge2'].fmeasure:.3f}")
        print(f"  TextRank ROUGE-L F1: {textrank_scores['rougeL'].fmeasure:.3f}")
    else:
        print(f"  No TextRank summary")
    
    # Calculate LexRank ROUGE scores
    lexrank_scores = None
    if lexrank_summary:
        lexrank_scores = calculate_rouge_scores(lexrank_summary, ground_truth)
        print(f"  LexRank ROUGE-1 F1: {lexrank_scores['rouge1'].fmeasure:.3f}")
        print(f"  LexRank ROUGE-2 F1: {lexrank_scores['rouge2'].fmeasure:.3f}")
        print(f"  LexRank ROUGE-L F1: {lexrank_scores['rougeL'].fmeasure:.3f}")
    else:
        print(f"  No LexRank summary")
    
    # Calculate DistilBART ROUGE scores
    distilbart_scores = None
    if distilbart_summary:
        distilbart_scores = calculate_rouge_scores(distilbart_summary, ground_truth)
        print(f"  DistilBART ROUGE-1 F1: {distilbart_scores['rouge1'].fmeasure:.3f}")
        print(f"  DistilBART ROUGE-2 F1: {distilbart_scores['rouge2'].fmeasure:.3f}")
        print(f"  DistilBART ROUGE-L F1: {distilbart_scores['rougeL'].fmeasure:.3f}")
    else:
        print(f"  No DistilBART summary")
    
    # Calculate Pegasus ROUGE scores
    pegasus_scores = None
    if pegasus_summary:
        pegasus_scores = calculate_rouge_scores(pegasus_summary, ground_truth)
        print(f"  Pegasus ROUGE-1 F1: {pegasus_scores['rouge1'].fmeasure:.3f}")
        print(f"  Pegasus ROUGE-2 F1: {pegasus_scores['rouge2'].fmeasure:.3f}")
        print(f"  Pegasus ROUGE-L F1: {pegasus_scores['rougeL'].fmeasure:.3f}")
    else:
        print(f"  No Pegasus summary")
    
    # Store results
    rouge_results.append({
        'ground_truth': gt_name,
        'textrank_rouge1': textrank_scores['rouge1'].fmeasure if textrank_scores else 0.0,
        'textrank_rouge2': textrank_scores['rouge2'].fmeasure if textrank_scores else 0.0,
        'textrank_rougeL': textrank_scores['rougeL'].fmeasure if textrank_scores else 0.0,
        'lexrank_rouge1': lexrank_scores['rouge1'].fmeasure if lexrank_scores else 0.0,
        'lexrank_rouge2': lexrank_scores['rouge2'].fmeasure if lexrank_scores else 0.0,
        'lexrank_rougeL': lexrank_scores['rougeL'].fmeasure if lexrank_scores else 0.0,
        'distilbart_rouge1': distilbart_scores['rouge1'].fmeasure if distilbart_scores else 0.0,
        'distilbart_rouge2': distilbart_scores['rouge2'].fmeasure if distilbart_scores else 0.0,
        'distilbart_rougeL': distilbart_scores['rougeL'].fmeasure if distilbart_scores else 0.0,
        'pegasus_rouge1': pegasus_scores['rouge1'].fmeasure if pegasus_scores else 0.0,
        'pegasus_rouge2': pegasus_scores['rouge2'].fmeasure if pegasus_scores else 0.0,
        'pegasus_rougeL': pegasus_scores['rougeL'].fmeasure if pegasus_scores else 0.0,
    })

print(f"\nROUGE score calculation completed for {len(rouge_results)} evaluations")


Calculating ROUGE scores for overall summaries...

Evaluating against gpt-4o-mini ground truth:
  TextRank ROUGE-1 F1: 0.209
  TextRank ROUGE-2 F1: 0.034
  TextRank ROUGE-L F1: 0.124
  LexRank ROUGE-1 F1: 0.248
  LexRank ROUGE-2 F1: 0.038
  LexRank ROUGE-L F1: 0.137
  DistilBART ROUGE-1 F1: 0.330
  DistilBART ROUGE-2 F1: 0.035
  DistilBART ROUGE-L F1: 0.165
  Pegasus ROUGE-1 F1: 0.293
  Pegasus ROUGE-2 F1: 0.012
  Pegasus ROUGE-L F1: 0.145

Evaluating against groq-llama ground truth:
  TextRank ROUGE-1 F1: 0.181
  TextRank ROUGE-2 F1: 0.018
  TextRank ROUGE-L F1: 0.133
  LexRank ROUGE-1 F1: 0.227
  LexRank ROUGE-2 F1: 0.013
  LexRank ROUGE-L F1: 0.133
  DistilBART ROUGE-1 F1: 0.302
  DistilBART ROUGE-2 F1: 0.021
  DistilBART ROUGE-L F1: 0.143
  Pegasus ROUGE-1 F1: 0.265
  Pegasus ROUGE-2 F1: 0.012
  Pegasus ROUGE-L F1: 0.118

ROUGE score calculation completed for 2 evaluations


In [6]:
# Create comparison DataFrame and display results
if rouge_results:
    df_rouge = pd.DataFrame(rouge_results)
    
    print("\nROUGE SCORES COMPARISON TABLE")
    print("=" * 120)
    
    # Display detailed results
    print(f"\nDetailed ROUGE Scores by Ground Truth:")
    print("-" * 120)
    print(f"{'Ground Truth':<15} {'Method':<12} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10}")
    print("-" * 120)
    
    for _, row in df_rouge.iterrows():
        gt_name = row['ground_truth']
        print(f"{gt_name:<15} {'TextRank':<12} {row['textrank_rouge1']:<10.3f} {row['textrank_rouge2']:<10.3f} {row['textrank_rougeL']:<10.3f}")
        print(f"{'':<15} {'LexRank':<12} {row['lexrank_rouge1']:<10.3f} {row['lexrank_rouge2']:<10.3f} {row['lexrank_rougeL']:<10.3f}")
        print(f"{'':<15} {'DistilBART':<12} {row['distilbart_rouge1']:<10.3f} {row['distilbart_rouge2']:<10.3f} {row['distilbart_rougeL']:<10.3f}")
        print(f"{'':<15} {'Pegasus':<12} {row['pegasus_rouge1']:<10.3f} {row['pegasus_rouge2']:<10.3f} {row['pegasus_rougeL']:<10.3f}")
        print("-" * 120)
    
    # Calculate average scores across all ground truth files
    print(f"\nAVERAGE ROUGE SCORES (across all ground truth files):")
    print("-" * 70)
    print(f"{'Method':<12} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'Total':<10}")
    print("-" * 70)
    
    textrank_avg_1 = df_rouge['textrank_rouge1'].mean()
    textrank_avg_2 = df_rouge['textrank_rouge2'].mean()
    textrank_avg_L = df_rouge['textrank_rougeL'].mean()
    textrank_total = textrank_avg_1 + textrank_avg_2 + textrank_avg_L
    
    lexrank_avg_1 = df_rouge['lexrank_rouge1'].mean()
    lexrank_avg_2 = df_rouge['lexrank_rouge2'].mean()
    lexrank_avg_L = df_rouge['lexrank_rougeL'].mean()
    lexrank_total = lexrank_avg_1 + lexrank_avg_2 + lexrank_avg_L
    
    distilbart_avg_1 = df_rouge['distilbart_rouge1'].mean()
    distilbart_avg_2 = df_rouge['distilbart_rouge2'].mean()
    distilbart_avg_L = df_rouge['distilbart_rougeL'].mean()
    distilbart_total = distilbart_avg_1 + distilbart_avg_2 + distilbart_avg_L
    
    pegasus_avg_1 = df_rouge['pegasus_rouge1'].mean()
    pegasus_avg_2 = df_rouge['pegasus_rouge2'].mean()
    pegasus_avg_L = df_rouge['pegasus_rougeL'].mean()
    pegasus_total = pegasus_avg_1 + pegasus_avg_2 + pegasus_avg_L
    
    print(f"{'TextRank':<12} {textrank_avg_1:<10.3f} {textrank_avg_2:<10.3f} {textrank_avg_L:<10.3f} {textrank_total:<10.3f}")
    print(f"{'LexRank':<12} {lexrank_avg_1:<10.3f} {lexrank_avg_2:<10.3f} {lexrank_avg_L:<10.3f} {lexrank_total:<10.3f}")
    print(f"{'DistilBART':<12} {distilbart_avg_1:<10.3f} {distilbart_avg_2:<10.3f} {distilbart_avg_L:<10.3f} {distilbart_total:<10.3f}")
    print(f"{'Pegasus':<12} {pegasus_avg_1:<10.3f} {pegasus_avg_2:<10.3f} {pegasus_avg_L:<10.3f} {pegasus_total:<10.3f}")
    
    # Determine winners
    print(f"\nPERFORMANCE COMPARISON:")
    print("-" * 50)
    
    all_scores_1 = {
        'TextRank': textrank_avg_1,
        'LexRank': lexrank_avg_1,
        'DistilBART': distilbart_avg_1,
        'Pegasus': pegasus_avg_1
    }
    all_scores_2 = {
        'TextRank': textrank_avg_2,
        'LexRank': lexrank_avg_2,
        'DistilBART': distilbart_avg_2,
        'Pegasus': pegasus_avg_2
    }
    all_scores_L = {
        'TextRank': textrank_avg_L,
        'LexRank': lexrank_avg_L,
        'DistilBART': distilbart_avg_L,
        'Pegasus': pegasus_avg_L
    }
    all_totals = {
        'TextRank': textrank_total,
        'LexRank': lexrank_total,
        'DistilBART': distilbart_total,
        'Pegasus': pegasus_total
    }
    
    rouge1_winner = max(all_scores_1, key=all_scores_1.get)
    rouge2_winner = max(all_scores_2, key=all_scores_2.get)
    rougeL_winner = max(all_scores_L, key=all_scores_L.get)
    overall_winner = max(all_totals, key=all_totals.get)
    
    print(f"ROUGE-1 Winner: {rouge1_winner} ({all_scores_1[rouge1_winner]:.3f})")
    print(f"ROUGE-2 Winner: {rouge2_winner} ({all_scores_2[rouge2_winner]:.3f})")
    print(f"ROUGE-L Winner: {rougeL_winner} ({all_scores_L[rougeL_winner]:.3f})")
    print(f"\nOverall Winner: {overall_winner} (Total: {all_totals[overall_winner]:.3f})")
    print(f"\nTotal Scores:")
    for model, score in sorted(all_totals.items(), key=lambda x: x[1], reverse=True):
        print(f"  {model}: {score:.3f}")
    
    # Save results to CSV
    df_rouge.to_csv(EVAL_OUTPUT_DIR / 'rouge_evaluation_results.csv', index=False)
    print(f"\nResults saved to: {EVAL_OUTPUT_DIR / 'rouge_evaluation_results.csv'}")
    
else:
    print("No ROUGE scores calculated. Please check if ground truth files are available.")



ROUGE SCORES COMPARISON TABLE

Detailed ROUGE Scores by Ground Truth:
------------------------------------------------------------------------------------------------------------------------
Ground Truth    Method       ROUGE-1    ROUGE-2    ROUGE-L   
------------------------------------------------------------------------------------------------------------------------
gpt-4o-mini     TextRank     0.209      0.034      0.124     
                LexRank      0.248      0.038      0.137     
                DistilBART   0.330      0.035      0.165     
                Pegasus      0.293      0.012      0.145     
------------------------------------------------------------------------------------------------------------------------
groq-llama      TextRank     0.181      0.018      0.133     
                LexRank      0.227      0.013      0.133     
                DistilBART   0.302      0.021      0.143     
                Pegasus      0.265      0.012      0.118     
--------